# Phase 1g — an ensemble of different losses

Phase 1f measured ranking quality free of any decision threshold, and the
result reversed a Phase 1d conclusion.

| loss | macro-F1 (Phase 1d ranking) | average precision on S |
|---|---|---|
| focal | **0.545** — 1st, and the one we kept | 0.285 — 4th |
| weighted | 0.492 | 0.256 |
| oversample | 0.484 | **0.388** — best |
| plain | 0.379 — last | 0.340 |

Focal won on a score that moves when the decision line moves. On the score
that does not, it ranks S beats *worse* than the two arms that lost.

**The current ensemble is five copies of focal — the weakest S ranker.**
An ensemble gains from members that make different mistakes, so this trains
fold models for all four losses and asks whether mixing them helps.

## Why the existing checkpoints cannot be reused

`plain_ce.pt`, `weighted_ce.pt` and `oversample.pt` were each fitted on **all
of DS1**, so no out-of-fold predictions exist for them. Choosing a composition
by comparing candidates on DS2 would be selecting on the test set — which
`thresholds.py` forbids and which would invalidate every DS2 number the
project has published. Fold models are retrained here so the choice can be
made on DS1 alone.

## The rule, fixed before the run

Five candidates, named in `COMPOSITIONS` in the script. The winner is chosen
by **pooled out-of-fold macro average precision**, and only then is DS2
scored. Candidate `E` drops focal entirely — that is a real hypothesis here,
not a courtesy option.

20 training runs. Roughly 10–20 minutes on a T4.

## 1. Environment and GPU

`CUBLAS_WORKSPACE_CONFIG` is set before anything imports torch — CUDA reads
it when it builds its context, and setting it later does nothing.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['SMART_HEALTH_DATA'] = '/content/physionet-data'

!nvidia-smi -L || echo 'NO GPU -- Runtime -> Change runtime type -> GPU'

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Change the runtime type before continuing.')
print('device:', torch.cuda.get_device_name(0))

## 2. Upload the code

Rebuild the archive on your laptop first — it must contain
`scripts/train_ensemble_mix.py`, which is new in this phase:

```
./venv/bin/python scripts/package_for_colab.py
```

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()
name = next(iter(uploaded))

work = pathlib.Path('/content/phase1g')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(name) as archive:
    archive.extractall(work)

%cd /content/phase1g
assert pathlib.Path('scripts/train_ensemble_mix.py').exists(), (
    'train_ensemble_mix.py missing -- rebuild the archive with '
    'scripts/package_for_colab.py and upload it again')
print('code ready')

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt
import numpy, scipy, sklearn, wfdb
print('numpy', numpy.__version__, '| sklearn', sklearn.__version__,
      '| wfdb', wfdb.__version__)

## 4. GATE — tests must pass before any GPU time is spent

405 tests, about 15 seconds.

In [ ]:
import subprocess
if subprocess.run(['python', '-m', 'pytest', '-q'], text=True).returncode != 0:
    raise SystemExit('Tests failed. Do not train until this is green.')
print('\ntests green')

## 5a. Download MIT-BIH (retried)

PhysioNet drops connections often enough that this is retried automatically.
The script is idempotent — a re-run fetches only what is missing.

In [ ]:
import subprocess

for attempt in range(1, 6):
    print(f'--- download attempt {attempt} ---', flush=True)
    subprocess.run(['python', 'scripts/download_mitbih.py'], text=True)
    check = subprocess.run(['python', 'scripts/download_mitbih.py', '--verify'],
                           text=True, capture_output=True)
    if check.returncode == 0:
        print('download verified')
        break
else:
    raise SystemExit('could not fetch MIT-BIH after 5 attempts')

## 5b. Preprocess

In [ ]:
!python scripts/preprocess_mitbih.py \
    --strategy ds1_ds2 --out-dir /content/physionet-data/processed/mitdb-ds12

## 5c. GATE — the data must match the earlier phases exactly

A missing record shrinks DS2 and makes every number incomparable to the
results already in `results/`.

In [ ]:
import json, pathlib, sys
sys.path.insert(0, 'src')
from ecg import config, dataset, splits

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
beats = out / 'beats'

n_npz = len(list(beats.glob('*.npz')))
assert n_npz == 48, f'expected 48 processed records, found {n_npz}'

ds2 = [r for r in splits.DS2_RECORDS if r not in config.PACED_RECORDS]
data = dataset.load_records(beats, ds2)
assert data.n_kept == 49646, f'DS2 has {data.n_kept} beats, expected 49646'
print(f'48 records, DS2 = {data.n_kept} beats. Matches Phases 1d/1e/1f.')

## 5d. GATE — the folds must be the ones the current ensemble used

This is the comparison's foundation. If the new members are trained on a
different patient split than the five `beat_focal` models already in
`models/`, then 'mixing helps' and 'the split changed' become impossible to
tell apart.

In [ ]:
from ecg import folds

EXPECTED = [
    ['118', '207', '208', '215', '223'],
    ['109', '115', '201', '203', '220'],
    ['106', '112', '114', '124'],
    ['119', '122', '205', '209'],
    ['101', '108', '116', '230'],
]

assignment = folds.make_folds(n_folds=5, seed=config.PHASE_1D_SEED)
assignment.validate()
made = [list(f) for f in assignment.folds]
assert made == EXPECTED, f'fold assignment changed:\n{made}'
print('fold assignment matches nn_folds.json from Phase 1e')

## 6. Run the study

4 losses × 5 folds = **20 training runs**, then composition selection on
out-of-fold predictions, then one pass over DS2.

Watch for the `[select]` block. It prints the winner and the line
`DS2 has not been touched yet.` — that ordering is the point of the whole
design, and the script enforces it rather than trusting the operator.

In [ ]:
!python scripts/train_ensemble_mix.py \
    --beats-dir /content/physionet-data/processed/mitdb-ds12/beats \
    --out-dir   /content/physionet-data/processed/mitdb-ds12 \
    --device cuda --epochs 50 --folds 5 \
    2>&1 | tee /content/phase1g-run.log

### If you want it shorter

Stop the cell above and run this instead. 20 epochs caps the budget rather
than letting early stopping decide, so a few folds may stop short of their
best — the comparison between compositions stays fair because the cap
applies to every arm equally.

In [ ]:
# !python scripts/train_ensemble_mix.py \
#     --beats-dir /content/physionet-data/processed/mitdb-ds12/beats \
#     --out-dir   /content/physionet-data/processed/mitdb-ds12 \
#     --device cuda --epochs 20 --folds 5 \
#     2>&1 | tee /content/phase1g-run.log

## 7. Read the headline before you leave

Two questions. Did selection on out-of-fold data pick the composition that
actually wins on DS2? And does the winner beat five copies of focal?

In [ ]:
import json, pathlib

m = json.loads(pathlib.Path(
    '/content/physionet-data/processed/mitdb-ds12/ensemble_mix_metrics.json'
).read_text())

selected = m['selection']['selected']
reference = m['selection']['reference']

print(f"{'composition':<28}{'OOF macroAP':>12}{'DS2 macroAP':>13}"
      f"{'DS2 S AP':>10}{'DS2 F1':>9}")
print('-' * 72)
for name in m['oof']['scores']:
    o = m['oof']['scores'][name]
    d = m['ds2']['scores'][name]
    tag = ' <- chosen' if name == selected else ''
    print(f"{name:<28}{o['macro_average_precision']:>12.4f}"
          f"{d['macro_average_precision']:>13.4f}"
          f"{d['s_average_precision']:>10.4f}"
          f"{d['headline_macro_f1']:>9.4f}{tag}")

best_on_ds2 = max(m['ds2']['scores'],
                  key=lambda k: m['ds2']['scores'][k]['macro_average_precision'])
print()
print('chosen on out-of-fold data :', selected)
print('best on DS2 in hindsight   :', best_on_ds2)
if selected != best_on_ds2:
    print('  -> selection cost something. Report the chosen one, not the best.')

w = m['ds2']['scores'][selected]
r = m['ds2']['scores'][reference]
print()
print(f"against {reference}: macro AP "
      f"{w['macro_average_precision'] - r['macro_average_precision']:+.4f}, "
      f"S AP {w['s_average_precision'] - r['s_average_precision']:+.4f}, "
      f"F1 {w['headline_macro_f1'] - r['headline_macro_f1']:+.4f}")

## 8. Collect results and weights

In [ ]:
import shutil, pathlib

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
bundle = pathlib.Path('/content/phase1g-results')
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / 'checkpoints_mix').mkdir(parents=True)

for name in ('ensemble_mix_metrics.json', 'nn_folds.json'):
    src = out / name
    if src.exists():
        shutil.copy(src, bundle / name)
        print('collected', name)
    else:
        print('MISSING', name)

shutil.copy('/content/phase1g-run.log', bundle / 'run.log')

n = 0
for ckpt in sorted((out / 'models').glob('mix_*.pt')):
    shutil.copy(ckpt, bundle / 'checkpoints_mix' / ckpt.name)
    n += 1
print(f'collected {n} checkpoints (expect 20)')

shutil.make_archive('/content/phase1g-results', 'zip', bundle)
size = pathlib.Path('/content/phase1g-results.zip').stat().st_size / 1024
print(f'/content/phase1g-results.zip  ({size:.0f} KB)')

### Save to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/phase1g-results.zip',
            '/content/drive/MyDrive/phase1g-results.zip')
print('saved to Drive')

### Or download

In [ ]:
from google.colab import files
files.download('/content/phase1g-results.zip')

## 9. Back on your laptop

```
./venv/bin/python scripts/ingest_colab_results.py ~/Downloads/phase1g-results.zip \
    --allow-arm-mismatch
```

`--allow-arm-mismatch` is needed because this bundle's arms are the four
losses, not Phase 1d's arm list — the ingest script checks that on purpose so
a stale bundle cannot quietly land in `results/`.